In [1]:
import pandas as pd 
import numpy as np
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
df = pd.read_csv("/kaggle/input/datasets/waseemalastal/imdb-dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
import re 
import string 
exclude = string.punctuation

def pre_process_data(text):
    
    #Lower text 
    text=str(text).lower().strip()
    
    #Remove Html tags
    pattern = re.compile('<.*?>')
    text = pattern.sub(r'',text)
    
    #Remove puctuation 
    text = text.translate(str.maketrans('','',exclude))

    #Sent tokenized document 
    return text.split(" ")

df["review"] = df["review"].apply(pre_process_data)

In [5]:
df["sentiment"] = df["sentiment"].apply(lambda x:0 if x == 'negative' else 1)
df.head()

,review,sentiment
0,"[one, of, the, other, reviewers, has, mentione...",1
1,"[a, wonderful, little, production, the, filmin...",1
2,"[i, thought, this, was, a, wonderful, way, to,...",1
3,"[basically, theres, a, family, where, a, littl...",0
4,"[petter, matteis, love, in, the, time, of, mon...",1


In [6]:
df["sentiment"].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

In [7]:
X = df.iloc[:,0]
y = df.iloc[:,1]

In [8]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=df['sentiment'])

In [9]:
y_train.value_counts()

sentiment
1    20000
0    20000
Name: count, dtype: int64

In [10]:
vocab = {'<PAD>':0,'<UNK>':1}

def build_vocab(row):
    for token in row:
        if token not in vocab:
            vocab[token] = len(vocab)
            
X_train.apply(build_vocab)

43863    None
49420    None
4942     None
49228    None
35522    None
         ... 
30223    None
48002    None
12986    None
20546    None
15197    None
Name: review, Length: 40000, dtype: object

In [11]:
len(vocab)
print(list(vocab.keys())[2:10])

['listening', 'to', 'the', 'soundtrack', 'at', 'moment', 'images', 'come']


In [12]:
def text_to_indices(text,vocab):
    tokens = text if isinstance(text,list) else pre_process_data(text)
    return [vocab.get(token,vocab['<UNK>']) for token in tokens]

In [13]:
class MyCustomDataset(Dataset):
    
    def __init__(self,X,y,vocab):
        self.features = X
        self.label= y
        self.vocab = vocab
        
    def __len__(self):
        return self.features.shape[0]
        
    def __getitem__(self,index):
        numeric_doc = text_to_indices(self.features[index],self.vocab)
        numeric_label = self.label[index]
        
        return torch.tensor(numeric_doc),torch.tensor(numeric_label)

In [14]:
train_dataset = MyCustomDataset(X_train.values,y_train.values,vocab)
test_dataset = MyCustomDataset(X_test.values, y_test.values, vocab)

In [15]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    docs,labels = zip(*batch)
    docs_padded = pad_sequence(docs,batch_first=True,padding_value=0)
    labels = torch.tensor(labels,dtype=torch.long)
    return docs_padded,labels
    
train_dataloader = DataLoader(train_dataset,batch_size=32,collate_fn=collate_fn)
test_dataloader= DataLoader(test_dataset,batch_size=32,collate_fn=collate_fn)

In [16]:
class  MyRNN(nn.Module):
    
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.GRU(50,64,batch_first=True)
        self.fc = nn.Linear(64,1)
        
    def forward(self,doc):
        embedded_doc = self.embedding(doc)
        output,hidden = self.rnn(embedded_doc)
        result = self.fc(hidden.squeeze(0))
        return result.squeeze(1)

In [34]:
learning_rate=0.001
epochs=20

In [35]:
model = MyRNN(len(vocab))
model.to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

In [36]:
for i in range(epochs):
    total_loss=0
    for batch_features,batch_labels in train_dataloader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        optimizer.zero_grad()
        output = model(batch_features)
        loss=criterion(output,batch_labels.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epochs:{i+1} Loss:{total_loss/len(train_dataloader)}")

Epochs:1 Loss:0.6950406244277955
Epochs:2 Loss:0.6920145575523377
Epochs:3 Loss:0.596739492559433
Epochs:4 Loss:0.30902313389778135
Epochs:5 Loss:0.19393840551674366
Epochs:6 Loss:0.12928497701361774
Epochs:7 Loss:0.08631151045002043
Epochs:8 Loss:0.06080184897799045
Epochs:9 Loss:0.041774483711179346
Epochs:10 Loss:0.028399390763044357
Epochs:11 Loss:0.023966067050630226
Epochs:12 Loss:0.013992051906418055
Epochs:13 Loss:0.010054793691332452
Epochs:14 Loss:0.007140796457428951
Epochs:15 Loss:0.006548265361203812
Epochs:16 Loss:0.006945497774000978
Epochs:17 Loss:0.00426421287539415
Epochs:18 Loss:0.0038414605296420634
Epochs:19 Loss:0.0037063082612512517
Epochs:20 Loss:0.002650816727860365


In [37]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_feature,batch_labels in test_dataloader:
        batch_feature,batch_labels = batch_feature.to(device),batch_labels.to(device)
        output = model(batch_feature)
        prob = torch.sigmoid(output)
        pred = (prob >= 0.5).long()
        total += batch_labels.shape[0]
        correct += (pred == batch_labels).sum().item()
print(f"Accuracy: {correct/total}")

Accuracy: 0.8771


In [38]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_feature,batch_labels in train_dataloader:
        batch_feature,batch_labels = batch_feature.to(device),batch_labels.to(device)
        output = model(batch_feature)
        prob = torch.sigmoid(output)
        pred = (prob >= 0.5).long()
        total += batch_labels.shape[0]
        correct += (pred == batch_labels).sum().item()
print(f"Accuracy: {correct/total}")

Accuracy: 0.999925
